# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [1]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [2]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/19 00:00:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/usr/local/lib/python3.9/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Define function for loading data

In [3]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [4]:
#export
def clean_data(df):
    from pyspark.sql.types import IntegerType, FloatType, TimestampType
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''
    
    # START YOUR CODE HERE ---------
    df = df.withColumn("passenger_count", col("passenger_count").cast(IntegerType())) \
           .withColumn("total_amount", col("total_amount").cast(FloatType())) \
           .withColumn("tip_amount", col("tip_amount").cast(FloatType())) \
           .withColumn("trip_distance", col("trip_distance").cast(FloatType())) \
           .withColumn("fare_amount", col("fare_amount").cast(FloatType())) \
           .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime"))) \
           .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))

    # END YOUR CODE HERE -----------
    
    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [5]:
#export
from pyspark.sql.functions import sum as _sum, avg

def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate
    '''
    
    # Filter out trips with same pickup and dropoff
    df_filtered = df.filter(col("PULocationID") != col("DOLocationID"))
    
    # Group by pickup/dropoff, aggregate total passengers and sum of total_amount
    df_grouped = df_filtered.groupBy("PULocationID", "DOLocationID") \
        .agg(
            _sum("passenger_count").alias("total_passenger_count"),
            _sum("total_amount").alias("total_amount_sum")
        )
    
    # Compute per person rate
    df_result = df_grouped.withColumn(
        "per_person_rate",
        col("total_amount_sum") / col("total_passenger_count")
    ).drop("total_amount_sum")
    
    # Sort by total_passenger_count descending, then per_person_rate descending
    df_result = df_result.orderBy(
        col("total_passenger_count").desc(),
        col("per_person_rate").desc()
    ).limit(10)
    
    return df_result

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [6]:
#export
from pyspark.sql.functions import round as _round

def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent
    '''
    
    # Filter trips with fare_amount > 2 and trip_distance > 0
    df_filtered = df.filter((col("fare_amount") > 2) & (col("trip_distance") > 0))
    
    # Compute tip percent
    df_filtered = df_filtered.withColumn(
        "tip_percent",
        (col("tip_amount") * 100 / col("fare_amount"))
    )
    
    # Round trip_distance up to nearest mile
    df_filtered = df_filtered.withColumn(
        "trip_distance",
        ceil(col("trip_distance"))
    )
    
    # Group by rounded trip_distance and compute average tip percent
    df_result = df_filtered.groupBy("trip_distance") \
        .agg(avg("tip_percent").alias("tip_percent"))
    
    # Sort descending tip_percent and take top 15
    df_result = df_result.orderBy(col("tip_percent").desc()).limit(15)
    
    return df_result


### Q1.4

Determine the average speed at different times of day.

In [7]:
#export
from pyspark.sql.functions import col, hour, avg, unix_timestamp

def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed
    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour
    '''
    
    # Compute trip duration in hours
    df = df.withColumn(
        "trip_duration_hours",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
    )

    # Add hour of pickup and AM/PM flags
    df = df.withColumn("hour_of_day", hour(col("tpep_pickup_datetime")))
    
    # AM trips: 0-11, PM trips: 12-23
    am_df = df.filter((col("hour_of_day") >= 0) & (col("hour_of_day") <= 11)) \
              .groupBy("hour_of_day") \
              .agg((avg(col("trip_distance")) / avg(col("trip_duration_hours"))).alias("am_avg_speed"))
    
    pm_df = df.filter((col("hour_of_day") >= 12) & (col("hour_of_day") <= 23)) \
              .withColumn("hour_of_day", (col("hour_of_day") - 12)) \
              .groupBy("hour_of_day") \
              .agg((avg(col("trip_distance")) / avg(col("trip_duration_hours"))).alias("pm_avg_speed"))
    
    # Join AM and PM on hour_of_day
    result_df = am_df.join(pm_df, on="hour_of_day", how="full_outer") \
                     .orderBy("hour_of_day")
    
    # Rename column to match autograder exactly
    result_df = result_df.withColumnRenamed("hour_of_day", "time_of_day")
    
    return result_df


## The below cells are for you to investigate your solutions and will not be graded

In [8]:
df = load_data()
df = clean_data(df)

In [9]:
common_pair(df).show()

+------------+------------+---------------------+------------------+
|PULocationID|DOLocationID|total_passenger_count|   per_person_rate|
+------------+------------+---------------------+------------------+
|         239|         238|                   62|  4.26274198870505|
|         237|         236|                   60| 4.482500068346659|
|         263|         141|                   52|3.4190384974846473|
|         161|         236|                   42| 5.368571440378825|
|         148|          79|                   42| 4.711904752822149|
|         142|         238|                   39|  5.05487182812813|
|         141|         236|                   37| 4.355675723101641|
|         239|         143|                   37| 4.252162224537617|
|         239|         142|                   35| 3.817714350564139|
|          79|         170|                   34| 6.394705884596881|
+------------+------------+---------------------+------------------+



In [10]:
distance_with_most_tip(df).show()

+-------------+------------------+
|trip_distance|       tip_percent|
+-------------+------------------+
|            1|17.129815971513313|
|            2|15.815527155632552|
|           17|15.796441782308916|
|           20| 15.11240992123345|
|            3|14.886705727113446|
|            6|14.579695131601051|
|            5|14.245405861990653|
|            4|13.831569507473274|
|            9|13.814476557648435|
|            8|12.072596772433315|
|           19|11.952632334985276|
|           10|11.880490518902954|
|            7| 10.80057562837643|
|           21|10.739019886973427|
|           18|10.696822158448429|
+-------------+------------------+



In [11]:
time_with_most_traffic(df).show()

+-----------+------------------+-------------------+
|time_of_day|      am_avg_speed|       pm_avg_speed|
+-----------+------------------+-------------------+
|          0| 9.377696196631234|               NULL|
|          1|10.845483413697353|  5.125214305177561|
|          3|              NULL|                0.0|
|          4|              NULL|                0.0|
|          5|              NULL| 0.5137660239764732|
|          6|              NULL|  9.989847870647605|
|          7|              NULL|0.18415305490417713|
|          8|              NULL| 0.5183127622697896|
|         10|              NULL| 0.6147483972627696|
|         11|              NULL|  4.650958285207579|
+-----------+------------------+-------------------+

